# 混合检索与失败修复

使用同一份手册和15个问题比较语义检索与混合检索。先核对原先漏答的R03、R04、R09，再运行其余12题。下面保留这次实验的输出。

## 检索方法

在语义检索之外增加2–4字片段的TF-IDF检索。两路各取最多8个候选，通过RRF融合排名，最后取3块。匹配时去掉空白，原文与来源保留；预期答案和页码用于事后核对。

In [3]:
try:
    import retrieval_upgrade as up
except ModuleNotFoundError:
    from public_manual import retrieval_upgrade as up

print(up.normalize_text("绿色模式/高效\n率模式"))
# 预期：绿色模式/高效率模式


绿色模式/高效率模式


## 准备资料与索引

安装仓库根目录依赖后，选择对应的Python内核。原文向量和TF-IDF索引会被后面的提问复用。对照使用 `llm08_baseline15.json` 中的旧回答，保存位置见 [README](../README.md#运行与查看结果)。

In [4]:
try:
    import manual_project as mp
except ModuleNotFoundError:
    from public_manual import manual_project as mp

records = mp.prepare_documents()
test_cases = mp.load_cases()
baseline = up.load_baseline()
model = mp.load_search_model()
doc_vectors = mp.encode_documents(records, model)
tools = up.build_search_tools(records, model, doc_vectors)

failed_ids = {"R03", "R04", "R09"}
failed_cases = [case for case in test_cases if case["id"] in failed_ids]
print("原文记录：", len(records), "旧结果：", len(baseline))


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

原文记录： 26 旧结果： 15


## 对比候选原文

先检查检索，不请求DeepSeek。声音报警的旧候选漏掉设置表，新候选包含第15页块01。

In [5]:
case = failed_cases[0]
old_hits = mp.retrieve(case["question"], records, doc_vectors, model, top_k=3)
new_hits = up.hybrid_find(case["question"], tools)
print("问题：", case["question"])
print("旧候选：", [hit["chunk_id"] for hit in old_hits])
print("新候选：", [hit["chunk_id"] for hit in new_hits])
up.show_hits(new_hits)


问题： 声音报警出厂时是开启还是关闭？
旧候选： ['p15-c03', 'p16-c04', 'p11-c02']
新候选： ['p15-c03', 'p15-c01', 'p16-c04']
EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块03
开。
如果配置为 0%，则在市电恢复后，
UPS 输出立即打开。
电池电量不足状
态指示设置
2 分钟 2 分钟、5 分钟、7 分钟、
10 分钟
当实际运行时间达到用户设置的限制
时，UPS 将发出声音警报。仅当 UPS
在电池模式下工作时声音警报才会发
出。

EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01
UPS 设置
使用显示界面配置 UPS 设置。请参阅“配置 UPS 参数”章节以编辑参数。
功能 出厂默认设置 用户可选择的选项 说明
输出电压 230 VAC 220、230、240 VAC 允许用户在 UPS 在线操作时选择输出
电压。
声音报警 启用 启用、禁用 当设置为“禁用”或按下静音按钮时，
UPS 的声音警报将静音。
绿色模式/高效
率模式
禁用 启用/禁用 当此模式启用时，只要输入电压处于
所配置输出电压的±5%范围内，和所
配置输出频率的±3 Hz 范围内，所连
设备即会通过旁路继电器接收市电输

EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第16页，块04
。按确认按钮可开始测试。
如果测试命令被接受，UPS 将开始自测并在显示屏上开始倒数。
测试结束时会显示信息。
测试被拒绝。输出关闭或电池不充电。
测试未通过
测试通过
内部原因导致测试终止
按静音/ESC 按钮返回主屏幕



### 核对具体依据

第15页块01中，声音报警的默认设置是启用。

In [ ]:
# 核对声音报警的默认设置
#EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01
#默认启用

## 三个漏检案例

模型、提示词、最终取3块都保持一致，只改检索方法。这格会请求DeepSeek三次。

In [6]:
from getpass import getpass
from openai import OpenAI

api_key = getpass("DeepSeek API key（隐藏输入）：").strip()
if not api_key:
    raise ValueError("请输入密钥")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com", timeout=60, max_retries=0)

fixed_results = up.run_cases(failed_cases, tools, client, "llm08_failed3.json")
up.compare_results(baseline, fixed_results)


R03 ok 声音报警出厂默认设置为“启用”。来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01：“声音报警 启用 启用、禁用”。 

R04 ok 不是。绿色模式/高效率模式的出厂默认设置为“禁用”。依据原文：“绿色模式/高效率模式 禁用”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01）。 

R09 ok 事件日志最多记录最后 10 个事件。  
依据：“UPS 记录最后 10 个事件和并在此日志中显示代码。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第16页，块01） 

R03 声音报警出厂时是开启还是关闭？
旧回答： 声音报警的出厂默认值：资料未提供。

手册摘录只提到：“当实际运行时间达到用户设置的限制时，UPS 将发出声音警报。仅当 UPS 在电池模式下工作时声音警报才会发出。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块03）

这可选项包括“2 分钟、5 分钟、7 分钟、10 分钟”，但未说明“声音报警”出厂时是开启还是关闭。
新回答： 声音报警出厂默认设置为“启用”。来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01：“声音报警 启用 启用、禁用”。
旧候选： ['p15-c03', 'p16-c04', 'p11-c02']
新候选： ['p15-c03', 'p15-c01', 'p16-c04']

R04 绿色或高效率模式默认开启吗？
旧回答： 资料未提供。摘录中没有关于“绿色或高效率模式”默认是否开启的说明。
新回答： 不是。绿色模式/高效率模式的出厂默认设置为“禁用”。依据原文：“绿色模式/高效率模式 禁用”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01）。
旧候选： ['p15-c03', 'p16-c04', 'p14-c03']
新候选： ['p15-c01', 'p16-c03', 'p15-c03']

R09 事件日志最多记录最后多少个事件？
旧回答： 资料未提供。手册摘录只说明如何查看日志和“在日志末尾”显示“End”，未给出事件日志的最大记录条数。


### 人工核对三条回答

以下评价对应保存的这次运行，依据是候选原文和实际回答。重跑后需要重新核对再保存。

In [8]:
# 三个漏答案例的人工评价

fixed_results[0]["review"] = "通过，依据PDF第15页，块01：“声音报警 启用 启用、禁用”"
fixed_results[1]["review"] = "通过，依据绿色模式/高效率模式 禁用”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01）"
fixed_results[2]["review"] = "通过，依据UPS 记录最后 10 个事件和并在此日志中显示代码。”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第16页，块01）"

mp.save_json(fixed_results, "llm08_failed3.json")

## 检查其余十二题

下一格请求DeepSeek十二次，合并保存15题记录。比较旧回答与新回答，检查检索方法变化后是否引入新的错误。

In [9]:
other_cases = [case for case in test_cases if case["id"] not in failed_ids]
other_results = up.run_cases(other_cases, tools, client, "llm08_other12.json")
by_id = {result["id"]: result for result in fixed_results + other_results}
all_results = [by_id[case["id"]] for case in test_cases]
mp.save_json(all_results, "llm08_all15.json")
up.compare_results(baseline, other_results)


R01 ok 手册中输出电压的出厂默认设置是 **230 VAC**。  
依据：功能“输出电压”的“出厂默认设置”列为“230 VAC”（《EasyUPS_SRV_Chinese_990-91063B.pdf》，PDF第15页，块01）。 

R02 ok 输出电压允许选择的值为 220、230、240 VAC。依据：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01] 中“用户可选择的选项”为“220、230、240 VAC”。 

R05 ok 不是。手册中“输出电压”的出厂默认设置为 **230 VAC**，不是 110V。  
依据：`功能 出厂默认设置 用户可选择的选项 说明`，其中“输出电压 230 VAC 220、230、240 VAC”。来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块01。 

R06 ok 不是。手册注：“PowerChute 是 64 位应用程序，无法安装在 32 位操作系统上。”（PDF第11页，块02；该句在块03中亦重复） 

R07 ok 不可以。手册明确说：“注：RS232 和 USB 无法同时使用。”来源：[EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块03]。 

R08 ok 手册给出的下载网址是：`www.apc.com/pcss`。原文：“请登录 www.apc.com/pcss 下载并安装最新版 PowerChute Serial Shutdown。”来源：EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第11页，块02。 

R10 ok 显示的剩余电池运行时间使用“分钟”。依据是手册中“剩余电池运行时间 分钟 BAT, 分”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第14页，块02）。 

R11 ok 输入频率范围为 40–70 Hz。依据原文：“频率 40 – 70 Hz”（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第5页，块01）。 

R12 ok 手册列出的电池类型是“密封式免维护阀控式铅酸电池 (SMF-VRLA)”。（EasyUPS_SRV_Chinese_99

### 本次观察

R03、R04、R09都找到了具体依据并答对。其余十二题未见明显退步，但没有逐条填写新的review。R15问当前实际电量，静态手册不能提供，回答“资料未提供”合理。

In [ ]:
# R03、R04、R09三处漏检得到修复。
# 其余十二题未见明显退步；密码与当前电量仍未编造。
# 本次是已参与调试的问题集和单次生成，需要新问题进一步检查。

## 结果文件

混合检索结果在 `results/llm08_all15.json`，对照基线在 `results/llm08_baseline15.json`。两份文件可以逐题查看旧回答、新回答和候选变化。